<a href="https://cloudevel.com"> <img src="img/cloudevel.png" width="500px"></a>

# Herramientas avanzadas de *Git*.

Este capítulo presenta dos herramientas de *Git* especialmente útiles en proyectos medianos y grandes:

* **`git bisect`**: permite localizar el *commit* exacto que introdujo un error mediante búsqueda binaria sobre el historial.
* **`git submodule`**: permite incrustar repositorios externos como subdirectorios de un repositorio principal, manteniendo sus historiales independientes.

## Preliminares.

A fin de contar con un entorno unificado, se utilizará una versión creada previamente del directorio ```demo``` que incluye los ejercicios de los capítulos previos y se encuentra comprimida en el archivo ```src/04/demo.zip```.

In [ ]:
rm -rf demo

In [ ]:
unzip -q src/04/demo.zip

In [ ]:
cd demo

In [ ]:
git log --oneline --graph

## El comando ```git bisect```.

El comando ```git bisect``` permite encontrar el *commit* exacto que introdujo un error (regresión) en el código, utilizando una **búsqueda binaria** sobre el historial de *commits*. En lugar de revisar cada *commit* uno por uno, *Git* divide el rango a la mitad en cada paso, reduciendo el número de revisiones necesarias a *O(log n)*.

```bash
git bisect <subcomando>
```

La documentación de referencia del comando ```git bisect``` está disponible en:

https://git-scm.com/docs/git-bisect

| Subcomando | Descripción |
|------------|-------------|
| ```start``` | Inicia la sesión de búsqueda binaria. |
| ```bad [commit]``` | Marca el *commit* actual (o el indicado) como **defectuoso**. |
| ```good [commit]``` | Marca un *commit* como **correcto** (sin el error). |
| ```skip``` | Omite el *commit* actual (p. ej., si no compila). |
| ```reset``` | Finaliza la sesión y regresa al *HEAD* original. |
| ```run <script>``` | Automatiza la búsqueda ejecutando un script en cada paso. |

### Flujo de trabajo con ```git bisect```.

El flujo básico consta de los siguientes pasos:

1. **Iniciar** la sesión de bisección.
2. **Marcar** el estado actual como defectuoso con ```git bisect bad```.
3. **Marcar** un *commit* antiguo conocido como correcto con ```git bisect good <id>```.
4. *Git* hace *checkout* del *commit* intermedio. Se **prueba** el código.
5. Según el resultado, se marca como ```bad``` o ```good```.
6. Se repite hasta que *Git* identifica el *commit* culpable.
7. **Finalizar** con ```git bisect reset```.

**Ejemplo:**

* Se inicia la sesión de bisección.

In [ ]:
git bisect start

* Se marca el estado actual (```HEAD```) como defectuoso.

In [ ]:
git bisect bad

* Se marca el primer *commit* del repositorio como correcto.

In [ ]:
git bisect good 2020250

* *Git* hace *checkout* del *commit* intermedio para que se pruebe. Según el resultado de la prueba, se marca como ```good``` o ```bad```. Este proceso se repite hasta que *Git* identifica el *commit* culpable.

In [ ]:
# Ejemplo: el commit intermedio es correcto
git bisect good

* Una vez identificado el *commit* culpable, se finaliza la sesión y se regresa al ```HEAD``` original.

In [ ]:
git bisect reset

### Automatización con ```git bisect run```.

Si se dispone de un *script* o comando que devuelve ```0``` cuando el código es correcto y un valor distinto de cero cuando es defectuoso, es posible automatizar toda la sesión:

```bash
git bisect start
git bisect bad HEAD
git bisect good <commit-bueno>
git bisect run <script-de-prueba>
git bisect reset
```

*Git* ejecutará el *script* en cada *commit* candidato y determinará automáticamente el culpable sin intervención manual.

## El comando ```git submodule```.

El comando ```git submodule``` permite incluir un repositorio *Git* externo como un subdirectorio de otro repositorio (el repositorio *padre*). El repositorio incluido se denomina **submódulo** y mantiene su propio historial independiente.

Esto resulta útil para:

* Reutilizar bibliotecas o componentes compartidos entre varios proyectos.
* Referenciar dependencias externas con una versión exacta y controlada.
* Organizar proyectos grandes en componentes con repositorios independientes.

```bash
git submodule <subcomando> <argumentos>
```

La configuración de los submódulos se almacena en el archivo ```.gitmodules``` en la raíz del repositorio padre.

La documentación de referencia del comando ```git submodule``` está disponible en:

https://git-scm.com/docs/git-submodule

### El comando ```git submodule add```.

Este comando agrega un repositorio externo como submódulo dentro del repositorio actual.

```bash
git submodule add <url> [ruta]
```

Donde:

* ```<url>``` es la *URL* del repositorio externo.
* ```[ruta]``` es el directorio donde se ubicará el submódulo. Si se omite, se usa el nombre del repositorio.

**Ejemplo:**

* La siguiente celda agrega el repositorio de la documentación de *Git* como submódulo.

In [ ]:
git submodule add https://github.com/git/git-scm.com libs/git-scm

In [ ]:
cat .gitmodules

### El comando ```git submodule status```.

Muestra el estado de cada submódulo: el *commit* al que apunta y si está inicializado.

```bash
git submodule status
```

El prefijo de cada línea indica el estado:

| Prefijo | Significado |
|---------|-------------|
| (sin prefijo) | Inicializado y sincronizado. |
| ```-``` | No inicializado (requiere ```git submodule init```). |
| ```+``` | El *commit* registrado difiere del *commit* actual en el submódulo. |
| ```U``` | El submódulo tiene conflictos de *merge*. |

In [ ]:
git submodule status

### Clonar un repositorio con submódulos.

Al clonar un repositorio que contiene submódulos, éstos no se descargan automáticamente. Existen dos formas de obtenerlos:

**Opción 1:** Clonar con submódulos en un solo paso.

```bash
git clone --recurse-submodules <url>
```

**Opción 2:** Inicializar y actualizar submódulos después del clon.

```bash
git clone <url>
git submodule init
git submodule update
```

### El comando ```git submodule update```.

Actualiza los submódulos para que apunten al *commit* registrado en el repositorio padre.

```bash
git submodule update [--init] [--remote] [--recursive]
```

| Opción | Descripción |
|--------|-------------|
| ```--init``` | Inicializa los submódulos que aún no lo están. |
| ```--remote``` | Actualiza al *commit* más reciente de la rama remota del submódulo (en lugar del *commit* registrado). |
| ```--recursive``` | Aplica la actualización a submódulos anidados. |

### El comando ```git submodule foreach```.

Ejecuta un comando arbitrario en cada submódulo del repositorio.

```bash
git submodule foreach <comando>
```

**Ejemplos:**

```bash
# Ver el estado de cada submódulo
git submodule foreach git status

# Actualizar la rama main en todos los submódulos
git submodule foreach git pull origin main
```

### Eliminar un submódulo.

No existe un comando directo para eliminar un submódulo. El proceso requiere varios pasos:

```bash
# 1. Eliminar la entrada del submódulo del índice y del directorio
git rm <ruta-del-submodulo>

# 2. Eliminar los metadatos del submódulo
rm -rf .git/modules/<ruta-del-submodulo>

# 3. Confirmar los cambios
git commit -m "Eliminar submódulo <nombre>"
```

## Referencias adicionales.

* https://git-scm.com/book/es/v2/Herramientas-de-Git-Depuraci%C3%B3n-con-Git
* https://git-scm.com/book/es/v2/Herramientas-de-Git-Subm%C3%B3dulos

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>